In [1]:
from langchain_community.document_compressors import infinity_rerank
from langgraph import graph
from langgraph.graph import START, END, StateGraph
import os
from typing import TypedDict
from dotenv import load_dotenv

In [6]:
class Batmasman(TypedDict):
    runs : int
    balls : int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundry_percentage : float
    summary :str


In [20]:
def calcualate_sr(state : Batmasman) -> Batmasman:
    sr = (state['runs']/state['balls'])*100
    return {'sr': sr}

def calculate_brb(state : Batmasman) -> Batmasman:
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'bpb': bpb}

def calculate_boundary_percentage(state : Batmasman) -> Batmasman:
    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/(state['runs'])*100)
    return {'boundry_percentage': boundary_percent}


def summary(state : Batmasman) -> Batmasman:
    summary = f"""
    Strike Rate - {state['sr']} \n
    Balls per boundary - {state['bpb']} \n
    Boundary percent - {state['boundry_percentage']}
    """

    return {'summary': summary}

In [21]:
graph = StateGraph(Batmasman)

graph.add_node("calcualate_sr", calcualate_sr)
graph.add_node("calculate_brb", calculate_brb)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("summary", summary)

graph.add_edge(START, "calcualate_sr")
graph.add_edge(START, "calculate_brb")
graph.add_edge(START, "calculate_boundary_percentage")

graph.add_edge("calcualate_sr", "summary")
graph.add_edge("calculate_brb", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()


In [24]:
initial_state = {"sixes" : 4, "fours" : 6, "runs": 100, "balls" : 50}
final_state = workflow.invoke(initial_state)
final_state

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundry_percentage': 48.0,
 'summary': '\n    Strike Rate - 200.0 \n\n    Balls per boundary - 5.0 \n\n    Boundary percent - 48.0\n    '}